# Variants: describing revisions with features

The Frankenstein notebooks contain about 1,500 **substitutions**: a word or phrase was deleted and replaced by another. For example:

> my utmost wonder was ~~caused by an~~ **engaged by some experiements** on an air pump

Questions
- **What kinds of revisions are there?** 
- **What is the role of Percy B. Shelley?**



## The data

Each row is one substitution. The columns used:

| Column | Content |
|---|---|
| `mod_id` | identifier of the revision |
| `notebook`, `folio`, `line_n` | position of the revision in the manuscript |
| `del_in_context` | the passage before the revision: about 10 words, the deleted reading, about 10 words |
| `add_in_context` | the same passage after the revision, with the added reading |
| `del_full`, `add_full` | only the deleted and the added reading, without context |
| `word_scope` | in the encoding: whether the revision affects a whole `word` or a `part-of-word` |
| `add_hand` | the hand of the addition (`#pbs` = Percy Bysshe Shelley) |

In [1]:
from pathlib import Path  # Provides tools for locating files and folders.
import pandas as pd  # Provides DataFrames for tables.
pd.set_option("display.max_colwidth", 120)  # Shows longer texts in table cells before cutting them with "...".

variants = pd.read_csv(Path("../data/Frankestein/variants-subst.csv"))  # Reads the spreadsheet: one row per substitution.
print(variants.shape)  # (number of variants, number of columns)

(1507, 21)


In [2]:
# One variant with its context: before and after the revision
example = variants.iloc[3]  # Takes one row of the table.
print("before:", example["del_in_context"])
print("after: ", example["add_in_context"])

before: early years to state those facts which led to my of that study. When I was eleven years old we all went on
after:  early years to state those facts which led to my predeliction for that science. When I was eleven years old we all went on


# Data cleaning

Not every row is a revision of the text. The following rows are removed:

- the `pagination` zone: page numbers corrected by the writers ("79" → "69");
- the `left_margin` zone: notes in the margin, outside the running text;
- revisions with `word_scope` = `uncertain`;
- passages that are identical before and after (ignoring spaces), for example a word overwritten with the same letters. There is nothing to measure there.

In [3]:
print(variants["zone"].value_counts())  # Where in the page the revisions are.
print(variants["word_scope"].value_counts())  # Whole word, part of a word, or uncertain.

# The passages that are identical before and after, once spaces are removed
identical = variants["del_in_context"].str.replace(" ", "") == variants["add_in_context"].str.replace(" ", "")
print(variants.loc[identical, ["del_full", "add_full", "del_rend"]])  # del_rend: how the deletion is marked. What happened here?

variants.loc[identical].to_csv("variants_identical-passages.csv", index=False)  # Saves these rows, to look at them more closely.
print("Saved to variants_identical-passages.csv")

zone
main           1482
left_margin      16
pagination        9
Name: count, dtype: int64
word_scope
word            1078
part-of-word     413
uncertain         16
Name: count, dtype: int64
        del_full    add_full                 del_rend
62           his         his              overwritten
113       Krempe      Krempe            strikethrough
156        cause       cause            strikethrough
423        facts       facts            strikethrough
463         that        that            strikethrough
654    emotio ns    emotions  strikethrough; unmarked
754        offer       offer            strikethrough
795      feeling     feeling            strikethrough
898          too         too            strikethrough
1288         had         had            strikethrough
1373      this .       this.  strikethrough; unmarked
1415  consummate  consummate            strikethrough
1479         saw         saw            strikethrough
1496       Felix       Felix              overwritten

In [4]:
keep = (variants["zone"] == "main") & (variants["word_scope"] != "uncertain") & ~identical  # ~ means "not".

columns = ["mod_id", "notebook", "folio", "line_n", "add_hand", "word_scope",
           "del_full", "add_full", "del_in_context", "add_in_context"]

selected = variants.loc[keep, columns].reset_index(drop=True)  # Only the rows to keep, only the necessary columns.

# A simpler column for the hand: the first hand listed, and "not specified" when empty.
# The encoding guidelines explain what an empty value means (most likely the main hand, Mary Shelley).
selected["hand"] = selected["add_hand"].fillna("not specified").str.split(";").str[0].str.strip()

print(len(variants), "rows before,", len(selected), "rows after")
print(selected["hand"].value_counts())

selected.to_csv("variants_selected.csv", index=False)  # Saves the selected rows and columns.
print("Saved to variants_selected.csv")

1507 rows before, 1453 rows after
hand
not specified    856
#pbs             593
#comp              3
#mws               1
Name: count, dtype: int64
Saved to variants_selected.csv


# Exercise 1: categories by hand

Before any computation, classify 50 variants by hand. 

1. Run the cell below, this will create a spreadsheet called `manual_classification.csv`
2. Open the file (e.g. in Excel or LibreOffice).
3. In the column `my_category`, write a category for each variant. The categories are free: there is no right answer.
4. Save the file (keeping the CSV format).

In [5]:
manual = selected.sample(50, random_state=42)[["mod_id", "del_in_context", "add_in_context", "del_full", "add_full"]].copy()  # 50 random variants.
manual["my_category"] = ""  # An empty column for the categories.
manual.to_csv("manual_classification.csv", index=False)  # Saves them to a file that can be edited.
print("Saved 50 variants to manual_classification.csv")

Saved 50 variants to manual_classification.csv


# Exercises 2: features by counting letters and words

Let's describe each revision **formally**, with a few measurements (called **features**):

For each revision, 5 columns are added, each measuring one formal aspect of the change:

| Feature | Meaning | Example |
|---|---|---|
| `n_removed` | number of words in the deleted reading (`del_full`) | "caused by an" → 3 |
| `n_added` | number of words in the added reading (`add_full`) | "engaged by some experiements" → 4 |
| `letter_similarity` | how similar the old and new reading are, letter by letter, based on the Levenshtein distance (0 = nothing in common, 1 = identical) | "examimations" → "examinations": 0.92 |
| `case_only` | 1 if only capitalisation changed | "the" → "The" |
| `same_lemma` | 1 if it is the same word in another form | "render" → "renders" |

Punctuation is not counted as words.

### Features 1 and 2. Words removed and added

The words of `del_full` and `add_full` are counted. Punctuation marks are not words, so they are not counted.

Note: when the transcription contains a space inside a word ("tho se"), the word is counted as two words.

In [6]:
import re  # Regular expressions, to find words in a text.

def words(text):
    # Returns the list of words in a text: sequences of letters or digits (with apostrophes, as in "don't"). Punctuation is left out.
    return re.findall(r"\w+(?:'\w+)*", text)

# Test on the example above
print(example["del_full"], "→", words(example["del_full"]))
print(example["add_full"], "→", words(example["add_full"]))

selected["n_removed"] = [len(words(text)) for text in selected["del_full"]]  # Number of words in the deleted reading.
selected["n_added"] = [len(words(text)) for text in selected["add_full"]]  # Number of words in the added reading.

print(selected[["del_full", "add_full", "n_removed", "n_added"]].head(10))

of that study. → ['of', 'that', 'study']
predeliction for that science. → ['predeliction', 'for', 'that', 'science']
                   del_full  \
0       caused by slight or   
1                      wish   
2                     their   
3            of that study.   
4                    Thonon   
5                       the   
6                     fined   
7  but continued to re with   
8                   parents   
9                      have   

                                                                                                                  add_full  \
0                                                                                               derive thier origin from a   
1                                                                                                                   desire   
2                                                                                                                therefore   
3                                    

### Feature 3. Letter similarity

The Levenshtein distance is the minimum number of single-letter edits needed to turn one reading into the other. There are three kinds of edits:
- insertion of a letter;
- deletion of a letter;
- substitution of a letter with another.

Examples: "examimations" → "examinations" = 1 (one substitution); "form" → "from" = 2; "wish" → "desire" = 5.

The distance depends on the length of the words: 2 edits are a lot for a word of 4 letters, but very few for a word of 15 letters. So the distance is divided by the length of the longer reading, and subtracted from 1:

letter_similarity = 1 − distance ÷ length of the longer reading

The result goes from 0 (nothing in common) to 1 (identical). Capitals and punctuation are ignored.

In [7]:
def levenshtein(a, b):
    # Returns the minimum number of insertions, deletions and substitutions of letters needed to turn a into b.
    previous = list(range(len(b) + 1))  # Distances between an empty text and the beginnings of b: 0, 1, 2, ...
    for i, letter_a in enumerate(a, start=1):  # Adds the letters of a one by one.
        current = [i]  # Distance between the first i letters of a and an empty text.
        for j, letter_b in enumerate(b, start=1):  # Compares with the letters of b one by one.
            current.append(min(
                previous[j] + 1,  # deletion of letter_a
                current[j - 1] + 1,  # insertion of letter_b
                previous[j - 1] + (letter_a != letter_b),  # substitution (costs 0 if the two letters are the same)
            ))
        previous = current
    return previous[-1]  # The distance between the whole of a and the whole of b.

def letter_similarity(text_a, text_b):
    # Turns the Levenshtein distance into a similarity from 0 to 1, ignoring capitals and punctuation.
    a = " ".join(words(text_a)).lower()
    b = " ".join(words(text_b)).lower()
    longest = max(len(a), len(b), 1)  # The length of the longer reading (at least 1, to avoid dividing by 0).
    return round(1 - levenshtein(a, b) / longest, 2)

# Test on some examples
print(levenshtein("examimations", "examinations"), letter_similarity("examimations", "examinations"))
print(levenshtein("form", "from"), letter_similarity("form", "from"))
print(levenshtein("wish", "desire"), letter_similarity("wish", "desire"))

selected["letter_similarity"] = [letter_similarity(a, b) for a, b in zip(selected["del_full"], selected["add_full"])]
print(selected[["del_full", "add_full", "letter_similarity"]].head(10))

1 0.92
2 0.5
5 0.17
                   del_full  \
0       caused by slight or   
1                      wish   
2                     their   
3            of that study.   
4                    Thonon   
5                       the   
6                     fined   
7  but continued to re with   
8                   parents   
9                      have   

                                                                                                                  add_full  \
0                                                                                               derive thier origin from a   
1                                                                                                                   desire   
2                                                                                                                therefore   
3                                                                                           predeliction for that science.   
4       

### Feature 4. Case only

The value is 1 if the two readings have the same letters, but different capitals ("the" → "The"). Punctuation is ignored.

In [8]:
case_only = []
for a, b in zip(selected["del_full"], selected["add_full"]):
    a = " ".join(words(a))  # The words of the deleted reading, without punctuation.
    b = " ".join(words(b))  # The words of the added reading, without punctuation.
    case_only.append(int(a != b and a.lower() == b.lower()))  # Different as written, identical in lower case. int() turns True/False into 1/0.
selected["case_only"] = case_only

print(selected.loc[selected["case_only"] == 1, ["del_full", "add_full"]].head(10))

       del_full    add_full
5           the         The
60         when        When
87         That        that
92          the         The
103         the         The
163         the         The
219           a           A
261  ingolstadt  Ingolstadt
299         the         The
311       about       About


### Feature 5. Same lemma

Here the passages in context (`del_in_context`, `add_in_context`) are used, because spaCy needs a sentence to analyse words correctly: in "I saw her" and "a saw", the lemma of "saw" depends on the context.

Both passages are lemmatised. If the sequence of lemmas is the same before and after the revision, but the words are different, the revision only changed the form of a word ("render" → "renders", "was" → "is").

In [9]:
import spacy  # Imports spaCy, as in session 1.

nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])  # Loads the English model; only tokens and lemmas are needed, so the slower parts are skipped.

# Analyse all passages with spaCy (nlp.pipe is faster than calling nlp() on each text; this takes a minute)
docs_before = list(nlp.pipe(selected["del_in_context"]))
docs_after = list(nlp.pipe(selected["add_in_context"]))

same_lemma = []
for before, after in zip(docs_before, docs_after):
    words_before = [token.text.lower() for token in before if not token.is_punct]  # The words of the passage, without punctuation.
    words_after = [token.text.lower() for token in after if not token.is_punct]
    lemmas_before = [token.lemma_.lower() for token in before if not token.is_punct]  # Their lemmas.
    lemmas_after = [token.lemma_.lower() for token in after if not token.is_punct]
    same_lemma.append(int(words_before != words_after and lemmas_before == lemmas_after))  # Different words, same lemmas.
selected["same_lemma"] = same_lemma

print(selected.loc[selected["same_lemma"] == 1, ["del_full", "add_full"]].head(10))

      del_full    add_full
53     pleased    pleasing
98    intended   intending
149   examined   examining
150   analyzed   analyzing
202        was        were
232  entreated  entreating
267   befitted   befitting
367      scale     scaling
566       felt     feeling
717        was        were


In [10]:
# Save the new table: the selected rows and columns, and the features
selected.to_csv("variants_features.csv", index=False)
print("Saved", len(selected), "variants to variants_features.csv")

Saved 1453 variants to variants_features.csv
